# AI Agents: chatbot batata hai, agent karta hai
**AI Explained · Episode 6 (full version)** · Jajati AI

Har cell ko upar se neeche chalao (Shift + Enter). Google Colab mein free chalta hai.

Hum ek **chhota agent** banayenge jo sabse sasti flight dhoondhta hai. Data nakli hai, koi website nahi khulti. Asli agent mein plan **LLM** banata hai aur tools asli hote hain (browser, search); yahan dono simple rakhe hain, taaki loop saaf dikhe: **Plan → Tools → Check**.

In [1]:
def chatbot(q):
    return ("1. Site kholo  2. Dates daalo  "
            "3. Khud compare karo")

print(chatbot("Goa ki sabse sasti flight?"))

1. Site kholo  2. Dates daalo  3. Khud compare karo


In [2]:
FLIGHTS = {   # nakli data
    "12 Dec": [("A101", 4890), ("B202", 5600)],
    "13 Dec": [("A105", 5150), ("C303", 6200)],
    "14 Dec": [("B208", 4990), ("C307", 5400)],
}
FULL = {"A101", "B208"}   # inmein seat nahi

def search(date):           # tool 1
    return FLIGHTS[date]

def seats(flight):          # tool 2
    return flight not in FULL

print(search("12 Dec"))

[('A101', 4890), ('B202', 5600)]


In [3]:
def plan(goal):
    # Asli agent mein ye plan LLM banata hai.
    steps = [("search", d) for d in goal["dates"]]
    return steps + [("pick", "sabse sasti"),
                    ("check", "seat hai?")]

goal = {"to": "Goa", "budget": 5500,
        "dates": ["12 Dec", "13 Dec", "14 Dec"]}
for step in plan(goal):
    print(step)

('search', '12 Dec')
('search', '13 Dec')
('search', '14 Dec')
('pick', 'sabse sasti')
('check', 'seat hai?')


In [4]:
def agent(goal, max_tries=5):
    found = []
    for step, arg in plan(goal):
        if step == "search":
            for f, p in search(arg):
                found.append((p, f, arg))
            print("TOOL search", arg)
    print(len(found), "flights mile")
    found.sort()   # sasti pehle
    for price, f, d in found[:max_tries]:
        print(f"PICK {f} ({d}) Rs {price}")
        if price > goal["budget"]:
            return "Budget mein kuch nahi"
        if seats(f):
            print("CHECK: seat hai")
            return f"{f}, {d}, Rs {price}"
        print("CHECK: FULL, phir se try")
    return "Nahi mila"

In [5]:
result = agent(goal)
print("JAWAB:", result)

TOOL search 12 Dec
TOOL search 13 Dec
TOOL search 14 Dec
6 flights mile
PICK A101 (12 Dec) Rs 4890
CHECK: FULL, phir se try
PICK B208 (14 Dec) Rs 4990
CHECK: FULL, phir se try
PICK A105 (13 Dec) Rs 5150
CHECK: seat hai
JAWAB: A105, 13 Dec, Rs 5150


In [6]:
def book(result, haan=False):
    if not haan:
        return "RUKO: payment se pehle tumhari haan"
    return "Booked (nakli): " + result

print(book(result))

RUKO: payment se pehle tumhari haan


### 3 safety rules
1. **Payment ya password se pehle agent ko roko.** Final click tumhara.
2. Web pages par likha text agent ko bhatka sakta hai (ise *prompt injection* kehte hain), isliye paise aur login wale kaam bina dekhe mat chhodo.
3. **Step limit rakho** (yahan `max_tries`), taaki loop hamesha chalta na rahe.

In [7]:
# Budget badlo, phir chalao:
goal["budget"] = 4500
print("JAWAB:", agent(goal))

TOOL search 12 Dec
TOOL search 13 Dec
TOOL search 14 Dec
6 flights mile
PICK A101 (12 Dec) Rs 4890
JAWAB: Budget mein kuch nahi
